# RAGantine D2: fine-tune the injection classifier (Colab, free GPU)

Fine-tunes `microsoft/deberta-v3-small` to flag instruction-like / poisoned chunks (SPEC §B6, D2 step 2).

**Inputs** (made locally with `uv run ragantine bench d2-dataset`, in `data/bench/`):
- `d2_train.jsonl`: positives are chunks of poison written **only from training templates t01-t04** on dev-split questions;
  negatives are random clean Wikipedia chunks (5x) plus hard negatives (imperative / how-to wording). Has a `split` column (train/val).
- `d2_dev.jsonl`: dev-split poison from the **test** templates t05-t08 plus clean chunks. Used **only** to pick the threshold.

Never train or tune on test-split questions. Outputs `models/d2-injection/` (weights + `threshold.json`) to download into the repo
(the folder is git-ignored), then run `uv run ragantine bench d2-threshold --write` locally to record the threshold in config.

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" scikit-learn sentencepiece

In [ ]:
# Upload d2_train.jsonl and d2_dev.jsonl (or mount Drive and point the paths at them).
import os
try:
    from google.colab import files  # type: ignore
    if not (os.path.exists("d2_train.jsonl") and os.path.exists("d2_dev.jsonl")):
        files.upload()
except ImportError:
    pass  # running locally: put the files next to the notebook
TRAIN_PATH, DEV_PATH = "d2_train.jsonl", "d2_dev.jsonl"

In [ ]:
SEED = 1337
BASE_MODEL = "microsoft/deberta-v3-small"
EPOCHS, LR, MAX_LEN, BATCH = 3, 2e-5, 512, 16
MAX_FALSE_QUARANTINE = 0.02          # threshold rule from SPEC §B6
USE_PUBLIC_DATASET = False           # deepset/prompt-injections (Apache-2.0): extra positives/negatives
OUT_DIR = "models/d2-injection"

In [ ]:
import json
from datasets import Dataset, concatenate_datasets, load_dataset

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

rows = read_jsonl(TRAIN_PATH)
assert all(r["template_id"] is None or int(r["template_id"][-2:]) <= 4 for r in rows), "test templates leaked into training data"
train = Dataset.from_list([{"text": r["text"], "label": r["label"]} for r in rows if r["split"] == "train"])
val = Dataset.from_list([{"text": r["text"], "label": r["label"]} for r in rows if r["split"] == "val"])
if USE_PUBLIC_DATASET:
    public = load_dataset("deepset/prompt-injections", split="train").select_columns(["text", "label"])
    train = concatenate_datasets([train, public.cast(train.features)])
dev = read_jsonl(DEV_PATH)
print(f"train={len(train)} val={len(val)} dev={len(dev)} positives(train)={sum(train['label'])}")

In [ ]:
import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

set_seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label={0: "SAFE", 1: "INJECTION"}, label2id={"SAFE": 0, "INJECTION": 1})

def encode(batch):
    return tok(batch["text"], truncation=True, max_length=MAX_LEN)

train_t, val_t = train.map(encode, batched=True), val.map(encode, batched=True)

def metrics(eval_pred):
    logits, labels = eval_pred
    p, r, f, _ = precision_recall_fscore_support(labels, logits.argmax(-1), average="binary", zero_division=0)
    return {"precision": p, "recall": r, "f1": f}

args = TrainingArguments(
    output_dir="d2-checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=torch.cuda.is_available(), seed=SEED, report_to=[], logging_steps=50)
trainer = Trainer(model=model, args=args, train_dataset=train_t, eval_dataset=val_t,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

In [ ]:
# Pick the threshold on the DEV split: highest recall with false-quarantine <= 2% on clean chunks.
@torch.no_grad()
def score(texts, bs=32):
    model.eval()
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(model.device)
        out += torch.softmax(model(**enc).logits, -1)[:, 1].tolist()
    return out

s = score([r["text"] for r in dev])
neg = sorted((x for x, r in zip(s, dev) if r["label"] == 0), reverse=True)
pos = [(x, r) for x, r in zip(s, dev) if r["label"] == 1]
allowed = int(MAX_FALSE_QUARANTINE * len(neg))
threshold = min(1.0, (neg[allowed] if allowed < len(neg) else -1.0) + 1e-6)
recall = sum(x >= threshold for x, _ in pos) / len(pos)
fq = sum(x >= threshold for x in neg) / len(neg)
by_attack = {}
for x, r in pos:
    by_attack.setdefault(r["attack"], []).append(x >= threshold)
print(f"threshold={threshold:.4f} recall={recall:.3f} false_quarantine={fq:.4f}")
print({a: round(sum(v) / len(v), 3) for a, v in sorted(by_attack.items())})

In [ ]:
import shutil
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
with open(f"{OUT_DIR}/threshold.json", "w") as f:
    json.dump({"threshold": threshold, "dev_recall": recall, "dev_false_quarantine": fq,
               "base_model": BASE_MODEL, "epochs": EPOCHS, "lr": LR, "max_len": MAX_LEN, "seed": SEED}, f, indent=2)
shutil.make_archive("d2-injection", "zip", OUT_DIR)
try:
    from google.colab import files  # type: ignore
    files.download("d2-injection.zip")  # unzip into <repo>/models/d2-injection/
except ImportError:
    print("saved to", OUT_DIR)
# Optional: model.push_to_hub("<you>/ragantine-d2-injection"); tok.push_to_hub(...)